# Embeddings con GPU (Colab T4) — RAG Banco de Occidente

Calcula los embeddings del corpus con la misma lógica del repositorio (`pipeline/fase3a/embed.py`), de forma **incremental**: solo codifica los fragmentos nuevos o modificados.

**Antes de empezar, sube a tu Drive una carpeta llamada `rag-bdo` con esta estructura (la misma del repositorio):**
```
MyDrive/rag-bdo/
├── rag-bocc/corpus.jsonl                 (archivo generado, ~65 MB; no está en git)
├── pipeline/fase3a/chunking.py
├── pipeline/fase3a/almacen.py
├── pipeline/fase3a/embed.py
└── _raw/emb/e5-small/                    (opcional: si ya existe, solo se recalcula lo que cambió)
```
Los resultados quedan en `rag-bdo/_raw/emb/<modelo>/`. **Descarga esas carpetas y cópialas en `_raw/emb/` de tu repositorio local.**

Activa la GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*. El corpus es información pública del sitio web del banco.

In [ ]:
# 1) Dependencias y GPU
!pip -q install sentence-transformers
import torch
print('CUDA disponible:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')

In [ ]:
# 2) Conectar Drive y comprobar los archivos
from google.colab import drive
drive.mount('/content/drive')
import os
RAIZ = '/content/drive/MyDrive/rag-bdo'
for f in ['rag-bocc/corpus.jsonl', 'pipeline/fase3a/chunking.py', 'pipeline/fase3a/almacen.py', 'pipeline/fase3a/embed.py']:
    p = f'{RAIZ}/{f}'; print('OK ' if os.path.exists(p) else 'FALTA', f)
os.environ['RAG_BASE'] = RAIZ

In [ ]:
# 3) Elige los modelos (e5-small es el de línea base; los otros son candidatos que en CPU tardan horas)
MODELOS = ['e5-small']          # opciones: 'e5-small', 'e5-large', 'bge-m3'
CTX = ['1', '0']                # 1 = con contexto de metadatos (el que se usa), 0 = sin contexto (para comparar)

In [ ]:
# 4) Calcular (incremental). Si el modelo ya tiene vectores en _raw/emb/<modelo>/, solo recodifica lo que cambió
for m in MODELOS:
    !python {RAIZ}/pipeline/fase3a/embed.py {m} --ctx {' '.join(CTX)} --dispositivo cuda

In [ ]:
# 5) Verificar: dimensiones, normas y manifiesto
import json, numpy as np, glob
for d in sorted(glob.glob(f'{RAIZ}/_raw/emb/*/')):
    mf = d + 'manifiesto.json'
    if not os.path.exists(mf): continue
    print(os.path.basename(d.rstrip('/')), json.dumps(json.load(open(mf)).get('calculos', {}), ensure_ascii=False)[:300])
    for f in sorted(glob.glob(d + '*.npy')):
        E = np.load(f); print('  ', os.path.basename(f), E.shape, 'norma media', round(float(np.linalg.norm(E, axis=1).mean()), 4), '| filas en cero:', int((np.abs(E).sum(1) == 0).sum()))
print('\nListo: descarga rag-bdo/_raw/emb/<modelo>/ y cópiala en _raw/emb/ del repositorio.')